# Add Continent Column to Renewable Energy Dataset

**Goal:** Enrich `renewable_energy_countries_selected.csv` with a `Continent` column by looking up each row's `iso_code` (ISO 3166-1 alpha-3) against the reference file `country-and-continent-codes-list-csv.csv`, which maps `Three_Letter_Country_Code` -> `Continent_Name`.

**Steps:**
1. Load both CSV files into pandas DataFrames.
2. Build a lookup table (ISO alpha-3 code -> Continent_Name) from the reference file.
3. Map the lookup onto the main dataset's `iso_code` column to create `Continent`.
4. Reorder columns so `Continent` sits right after `country`.
5. Identify and report any `iso_code` values that could not be matched, filling them with `"Unknown"`.
6. Save the result to `renewable_energy_countries_selected_v7.csv` and print a preview + match summary.

## Step 1: Imports and load the two CSV files

In [108]:
import pandas as pd

# File paths (same directory as this notebook)
REFERENCE_FILE = "country-and-continent-codes-list-csv.csv"
MAIN_FILE = "renewable_energy_countries_selected.csv"
OUTPUT_FILE = "renewable_energy_countries_selected_v7.csv"

# Load the reference file (continent / country code lookup table)
reference_df = pd.read_csv(REFERENCE_FILE, encoding="utf-8")

# Load the main renewable energy dataset
main_df = pd.read_csv(MAIN_FILE, encoding="utf-8")

print(f"Reference file shape: {reference_df.shape}")
print(f"Main dataset shape:   {main_df.shape}")

reference_df.head()

Reference file shape: (258, 6)
Main dataset shape:   (5381, 22)


,Continent_Name,Continent_Code,Country_Name,Two_Letter_Country_Code,Three_Letter_Country_Code,Country_Number
0,Asia,AS,"Afghanistan, Islamic Republic of",AF,AFG,4
1,Europe,EU,"Albania, Republic of",AL,ALB,8
2,Antarctica,AN,Antarctica (the territory South of 60 deg S),AQ,ATA,10
3,Africa,AF,"Algeria, People's Democratic Republic of",DZ,DZA,12
4,Oceania,OC,American Samoa,AS,ASM,16


## Step 2: Build a Three_Letter_Country_Code -> Continent_Name lookup

Some three-letter codes can legitimately appear more than once in reference lists (e.g. a country split across multiple continent-adjacent territories). We drop duplicate codes, keeping the first occurrence, so the lookup stays a clean 1-to-1 mapping. We also strip whitespace and uppercase the codes on both sides to avoid silent mismatches from formatting differences.

In [109]:
# Normalize the code columns (strip whitespace, uppercase) on a copy so matching is robust
ref_clean = reference_df.copy()
ref_clean["Three_Letter_Country_Code"] = (
    ref_clean["Three_Letter_Country_Code"].astype(str).str.strip().str.upper()
)

# Check for duplicate codes in the reference file before building the lookup
dup_codes = ref_clean[ref_clean.duplicated("Three_Letter_Country_Code", keep=False)]
if not dup_codes.empty:
    print("Warning: duplicate Three_Letter_Country_Code values found in reference file "
          "(keeping first occurrence for each):")
    display(dup_codes.sort_values("Three_Letter_Country_Code"))

# Build the lookup dict, keeping the first occurrence of each code
code_to_continent = (
    ref_clean.drop_duplicates("Three_Letter_Country_Code", keep="first")
    .set_index("Three_Letter_Country_Code")["Continent_Name"]
    .to_dict()
)

# ลบ key ที่เป็นค่าว่าง/NaN ออกจาก lookup กันไม่ให้ iso_code ว่างไปจับคู่มั่ว
code_to_continent.pop("", None)
code_to_continent.pop("NAN", None)

print(f"\nLookup built with {len(code_to_continent)} unique country codes.")

,Continent_Name,Continent_Code,Country_Name,Two_Letter_Country_Code,Three_Letter_Country_Code,Country_Number
16,Europe,EU,"Armenia, Republic of",AM,ARM,51
17,Asia,AS,"Armenia, Republic of",AM,ARM,51
8,Europe,EU,"Azerbaijan, Republic of",AZ,AZE,31
9,Asia,AS,"Azerbaijan, Republic of",AZ,AZE,31
58,Europe,EU,"Cyprus, Republic of",CY,CYP,196
59,Asia,AS,"Cyprus, Republic of",CY,CYP,196
83,Europe,EU,Georgia,GE,GEO,268
84,Asia,AS,Georgia,GE,GEO,268
116,Europe,EU,"Kazakhstan, Republic of",KZ,KAZ,398
117,Asia,AS,"Kazakhstan, Republic of",KZ,KAZ,398



Lookup built with 250 unique country codes.


## Step 3: Map `iso_code` to `Continent`

We normalize `iso_code` the same way as the reference codes before mapping, so casing/whitespace differences don't cause false mismatches.

In [110]:
result_df = main_df.copy()

# Normalize iso_code for matching (keep original iso_code column untouched in the output)
iso_normalized = result_df["iso_code"].astype(str).str.strip().str.upper()

# ป้องกันไม่ให้ iso_code ที่เป็นค่าว่าง/NaN ไปจับคู่กับแถวว่างในไฟล์ reference
iso_normalized = iso_normalized.replace({"": pd.NA, "NAN": pd.NA})

# Map to Continent via the lookup; unmatched codes become NaN for now
result_df["Continent"] = iso_normalized.map(code_to_continent)

## Step 4: Handle unmatched ISO codes

Any `iso_code` that has no corresponding entry in the reference file will have `Continent` as `NaN` after the mapping. We report exactly which codes (and how many rows) were unmatched, then fill those with `"Unknown"` so the column has no missing values.

In [111]:
unmatched_mask = result_df["Continent"].isna()
unmatched_codes = sorted(result_df.loc[unmatched_mask, "iso_code"].dropna().unique().tolist())

n_matched = (~unmatched_mask).sum()
n_unmatched = unmatched_mask.sum()

print(f"Rows matched:   {n_matched} / {len(result_df)}")
print(f"Rows unmatched: {n_unmatched} / {len(result_df)}")

if unmatched_codes:
    print(f"\nUnique iso_code values with no match in the reference file ({len(unmatched_codes)}):")
    for code in unmatched_codes:
        n_rows = (result_df["iso_code"] == code).sum()
        print(f"  - {code!r}: {n_rows} row(s)")
else:
    print("\nAll iso_code values were successfully matched to a continent.")

# Fill unmatched continents with "Unknown" so the column has no missing values
result_df["Continent"] = result_df["Continent"].fillna("Unknown")

Rows matched:   5355 / 5381
Rows unmatched: 26 / 5381

All iso_code values were successfully matched to a continent.


## Step 5: Reorder columns so `Continent` follows `country`

In [112]:
cols = list(result_df.columns)
cols.remove("Continent")
insert_at = cols.index("country") + 1
cols.insert(insert_at, "Continent")

result_df = result_df[cols]

print("Final column order:")
print(list(result_df.columns))

Final column order:
['country', 'Continent', 'year', 'iso_code', 'gdp', 'electricity_generation', 'renewables_share_elec', 'fossil_share_elec', 'low_carbon_share_elec', 'solar_electricity', 'wind_electricity', 'hydro_electricity', 'nuclear_electricity', 'coal_electricity', 'gas_electricity', 'oil_electricity', 'other_renewable_electricity', 'solar_share_elec', 'wind_share_elec', 'hydro_share_elec', 'nuclear_share_elec', 'coal_share_elec', 'gas_share_elec']


## Step 6: Preview, save, and summarize

In [113]:
# Preview the final DataFrame
result_df.head()

,country,Continent,year,iso_code,gdp,electricity_generation,renewables_share_elec,fossil_share_elec,low_carbon_share_elec,solar_electricity,...,coal_electricity,gas_electricity,oil_electricity,other_renewable_electricity,solar_share_elec,wind_share_elec,hydro_share_elec,nuclear_share_elec,coal_share_elec,gas_share_elec
0,Afghanistan,Asia,2000,AFG,1.128379e+10,0.48,64.583,35.417,64.583,0.0,...,0.00,0.0,0.17,0.0,0.0,0.0,64.583,0.0,0.000,0.0
1,Afghanistan,Asia,2001,AFG,1.102127e+10,0.69,72.464,27.536,72.464,0.0,...,0.04,0.0,0.15,0.0,0.0,0.0,72.464,0.0,5.797,0.0
2,Afghanistan,Asia,2002,AFG,1.880487e+10,0.71,78.873,21.127,78.873,0.0,...,0.04,0.0,0.11,0.0,0.0,0.0,78.873,0.0,5.634,0.0
3,Afghanistan,Asia,2003,AFG,2.107434e+10,0.91,69.231,30.769,69.231,0.0,...,0.09,0.0,0.19,0.0,0.0,0.0,69.231,0.0,9.890,0.0
4,Afghanistan,Asia,2004,AFG,2.233257e+10,0.79,70.886,29.114,70.886,0.0,...,0.06,0.0,0.17,0.0,0.0,0.0,70.886,0.0,7.595,0.0


In [114]:
# Save to CSV: comma-separated, UTF-8, no pandas index column
result_df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")

print(f"Saved {len(result_df)} rows and {len(result_df.columns)} columns to '{OUTPUT_FILE}'.")
print("\n--- Match summary ---")
print(f"Matched rows:   {n_matched} ({n_matched / len(result_df):.1%})")
print(f"Unmatched rows: {n_unmatched} ({n_unmatched / len(result_df):.1%})")
print(f"Unique unmatched iso_code values: {len(unmatched_codes)}")

Saved 5381 rows and 23 columns to 'renewable_energy_countries_selected_v7.csv'.

--- Match summary ---
Matched rows:   5355 (99.5%)
Unmatched rows: 26 (0.5%)
Unique unmatched iso_code values: 0
